In [ ]:
import os
import sys
import json
import re
import html
import argparse
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict, Any, Callable
import joblib
import numpy as np
import pandas as pd
import sqlite3
from dotenv import load_dotenv

load_dotenv()

# Pythainlp components
try:
    from pythainlp import word_tokenize
    from pythainlp.corpus import thai_stopwords
    from pythainlp.util import normalize as thai_normalize
    PYTHAINLP_AVAILABLE = True
except ImportError:
    PYTHAINLP_AVAILABLE = False
    print("Warning: pythainlp not installed. Thai tokenization will use basic splitting.")

# Gensim components
try:
    from gensim.models import Word2Vec
    GENSIM_AVAILABLE = True
except ImportError:
    GENSIM_AVAILABLE = False

# Transformers and Torch components
try:
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        EarlyStoppingCallback,
        AutoModel
    )
    from datasets import Dataset
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("Warning: transformers/torch not installed. BERT training unavailable.")

# Sklearn components
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, precision_recall_fscore_support
)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# --- Configuration Constants (Copied from sXDSV1FM9QQT) ---
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

FLOOD_HASHTAGS: List[str] = [
    "น้ำท่วม68", "ขอความช่วยเหลือ", "น้ำท่วม", "อุทกภัย",
    "สงขลา", "น้ำท่วมหาดใหญ่", "ภัยพิบัติภาคใต้", "น้ำท่วม2568",
    "ช่วยด้วย", "ขอความช่วยเหลือด่วน",
]
URGENCY_KEYWORDS: List[str] = [
    "ด่วน", "ช่วยด้วย", "SOS", "ขออพยพ", "ติดอยู่", "หลังคา",
    "จมน้ำ", "ช็อก", "เสียชีวิต", "บาดเจ็บ", "พิการ", "ท้อง",
    "ผู้สูงอายุ", "เด็ก", "ทารก",
]
FB_ACCESS_TOKEN = os.environ.get("FB_ACCESS_TOKEN", "")
DATA_DIR = "data"
MODELS_DIR = "models"
DB_PATH = os.path.join(DATA_DIR, "flood_posts.db")
JSONL_PATH = os.path.join(DATA_DIR, "raw_posts.jsonl")
URLS_FILE = os.path.join(DATA_DIR, "urls.txt")
RAW_CSV_PATH = os.path.join(DATA_DIR, "all_posts_raw.csv")
LABELED_CSV_PATH = os.path.join(DATA_DIR, "all_posts_labeled.csv")
TRAIN_CSV_PATH = os.path.join(DATA_DIR, "Train.csv")
TEST_CSV_PATH = os.path.join(DATA_DIR, "Test.csv")
SOS_JSON_PATH = os.path.join(DATA_DIR, "sos.json")
BERT_MODEL_DIR = os.path.join(MODELS_DIR, "bert_flood_model")
TFIDF_VEC_PATH = os.path.join(MODELS_DIR, "tfidf_vectorizer.joblib")
SVM_MODEL_PATH = os.path.join(MODELS_DIR, "svm_tfidf.joblib")
W2V_MODEL_PATH = os.path.join(MODELS_DIR, "word2vec.model")
THAI_BERT_MODELS = {
    "wangchanberta": "airesearch/wangchanberta-base-wiki-newmm",
    "wangchanberta_att": "airesearch/wangchanberta-base-att-spm-uncased",
    "phayathaibert": "clicknext/phayathaibert",
}
DEFAULT_BERT_MODEL = "airesearch/wangchanberta-base-att-spm-uncased"

@dataclass
class PreprocessConfig:
    clean: bool = True
    normalize: bool = True
    remove_stopwords: bool = True
    stemming: bool = False
    lemmatize: bool = False

    def __str__(self):
        flags = []
        if self.clean:
            flags.append("clean")
        if self.normalize:
            flags.append("norm")
        if self.remove_stopwords:
            flags.append("stop")
        if self.stemming:
            flags.append("stem")
        if self.lemmatize:
            flags.append("lemma")
        return "_".join(flags) if flags else "raw"

PREPROCESSING_PIPELINES = {
    "pipeline_1": PreprocessConfig(clean=False, normalize=False, remove_stopwords=False),
    "pipeline_2": PreprocessConfig(clean=True, normalize=False, remove_stopwords=False),
    "pipeline_3": PreprocessConfig(clean=True, normalize=True, remove_stopwords=False),
    "pipeline_4": PreprocessConfig(clean=True, normalize=True, remove_stopwords=True),
    "pipeline_5": PreprocessConfig(clean=True, normalize=True, remove_stopwords=True, stemming=True),
    "pipeline_6": PreprocessConfig(clean=True, normalize=True, remove_stopwords=True, lemmatize=True),
}

@dataclass
class TrainingConfig:
    test_size: float = 0.2
    random_seed: int = 42
    bert_model_name: str = DEFAULT_BERT_MODEL
    max_length: int = 256
    learning_rate: float = 1.5e-5
    batch_size: int = 16
    num_epochs: int = 4
    weight_decay: float = 0.05
    gradient_accumulation_steps: int = 2
    warmup_ratio: float = 0.05
    logging_steps: int = 50
    eval_strategy: str = "epoch"
    save_strategy: str = "epoch"
    load_best_model_at_end: bool = True
    metric_for_best_model: str = "f1"
    greater_is_better: bool = True
    fp16: bool = True
    bf16: bool = False
    dataloader_num_workers: int = 4
    w2v_vector_size: int = 200
    w2v_window: int = 5
    w2v_min_count: int = 2

# --- Preprocessing Utilities (Copied from lVix8n2R9QQT) ---
def get_thai_stopwords() -> set:
    if PYTHAINLP_AVAILABLE:
        return set(thai_stopwords())
    else:
        return {
            "และ", "หรือ", "แต่", "ที่", "ของ", "ใน", "เป็น", "มี", "ได้",
            "ไม่", "ก็", "จะ", "กับ", "ให้", "ว่า", "แล้ว", "นี้", "นั้น",
            "คือ", "จาก", "เมื่อ", "ถ้า", "อยู่", "กัน", "ครับ", "ค่ะ",
            "นะ", "คะ", "หน่อย", "ด้วย", "เลย", "มาก", "น่า", "ๆ",
        }
THAI_STOPWORDS = get_thai_stopwords()
def remove_urls(text: str) -> str:
    return re.sub(r'http[s]?://\S+', ' ', text)
def remove_emojis(text: str) -> str:
    return re.sub(r'[^\u0E00-\u0E7Fa-zA-Z0-9\s.,:;!?/-]', ' ', text)
def remove_hashtag_symbols(text: str) -> str:
    return re.sub(r'[#@]', '', text)
def normalize_whitespace(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()
def remove_repeated_chars(text: str, max_repeat: int = 2) -> str:
    pattern = r'(.)\1{' + str(max_repeat) + r',}'
    replacement = r'\1' * max_repeat
    return re.sub(pattern, replacement, text)
def normalize_thai_digits(text: str) -> str:
    thai_digits = '๐๑๒๓๔๕๖๗๘๙'
    arabic_digits = '0123456789'
    trans_table = str.maketrans(thai_digits, arabic_digits)
    return text.translate(trans_table)
def basic_clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = remove_urls(text)
    text = remove_emojis(text)
    text = remove_hashtag_symbols(text)
    text = normalize_whitespace(text)
    return text
def tokenize_thai(text: str, engine: str = "newmm") -> List[str]:
    if PYTHAINLP_AVAILABLE:
        return word_tokenize(text, engine=engine)
    else:
        return text.split()
def normalize_tokens(tokens: List[str]) -> List[str]:
    normalized = []
    for tok in tokens:
        tok = remove_repeated_chars(tok)
        tok = normalize_thai_digits(tok)
        tok = tok.replace(',', '')
        if tok.strip():
            normalized.append(tok)
    return normalized
def remove_stopwords(tokens: List[str], stopwords: set = None) -> List[str]:
    if stopwords is None:
        stopwords = THAI_STOPWORDS
    return [t for t in tokens if t not in stopwords]
def thai_stem(tokens: List[str]) -> List[str]:
    stemmed = []
    for tok in tokens:
        tok = tok.replace('ๆ', '')
        if tok.endswith('การ'):
            tok = tok[:-3] if len(tok) > 3 else tok
        stemmed.append(tok)
    return [t for t in stemmed if t.strip()]
def thai_lemmatize(tokens: List[str]) -> List[str]:
    if PYTHAINLP_AVAILABLE:
        try:
            from pythainlp.corpus import wordnet
            lemmatized = []
            for tok in tokens:
                synsets = wordnet.synsets(tok)
                if synsets:
                    lemma = synsets[0].lemma_names()[0]
                    lemmatized.append(lemma)
                else:
                    lemmatized.append(tok)
            return lemmatized
        except:
            return tokens
    return tokens
def preprocess_text(text: str, config: PreprocessConfig) -> List[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    if config.clean:
        text = basic_clean(text)
    if config.normalize:
        tokens = tokenize_thai(text)
        tokens = normalize_tokens(tokens)
    else:
        tokens = text.split()
    if config.remove_stopwords:
        tokens = remove_stopwords(tokens)
    if config.stemming:
        tokens = thai_stem(tokens)
    elif config.lemmatize:
        tokens = thai_lemmatize(tokens)
    tokens = [t for t in tokens if t.strip()]
    return tokens
def preprocess_text_to_string(text: str, config: PreprocessConfig) -> str:
    tokens = preprocess_text(text, config)
    return ' '.join(tokens)

# --- BERT Model Training Functions (Copied from p-AM8a5J9QQX) ---
def print_header(title: str):
    print("\n" + "=" * 60)
    print(f" {title}")
    print("=" * 60)

def load_data_for_bert(
    train_path: str = TRAIN_CSV_PATH,
    test_path: str = TEST_CSV_PATH,
    label_column: str = "label",
    multi_label: bool = False,
    label_list: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    train_df = pd.read_csv(train_path, encoding='utf-8-sig')
    test_df = pd.read_csv(test_path, encoding='utf-8-sig')

    train_df = train_df[['text', label_column]].dropna(subset=['text'])
    test_df = test_df[['text', label_column]].dropna(subset=['text'])

    label_names: List[str] = []

    if multi_label:
        # Placeholder for multi-label logic if needed in the future
        raise NotImplementedError("Multi-label not implemented for this function in this context.")
    else:
        train_df = train_df[['text', label_column]].dropna()
        test_df = test_df[['text', label_column]].dropna()
        if pd.api.types.is_numeric_dtype(train_df[label_column]):
            train_labels = train_df[label_column].astype(int)
            test_labels = test_df[label_column].astype(int)
            label_names = sorted(train_labels.unique().tolist())
        else:
            train_labels = train_df[label_column].astype(str)
            test_labels = test_df[label_column].astype(str)
            label_names = sorted(train_labels.unique().tolist())
            label_map = {name: idx for idx, name in enumerate(label_names)}
            train_labels = train_labels.map(label_map)
            test_labels = test_labels.map(label_map)
        train_df = pd.DataFrame({'text': train_df['text'], 'labels': train_labels})
        test_df = pd.DataFrame({'text': test_df['text'], 'labels': test_labels})

    print(f"Training samples: {len(train_df)}")
    print(f"Test samples: {len(test_df)}")
    print(f"Label distribution (train): {pd.Series(train_df['labels']).value_counts().to_dict()}")

    return train_df, test_df, label_names

def create_datasets(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    tokenizer,
    max_length: int = 256
) -> Tuple[Dataset, Dataset]:
    train_dataset = Dataset.from_pandas(train_df[['text', 'labels']])
    test_dataset = Dataset.from_pandas(test_df[['text', 'labels']])

    def tokenize_function(examples):
        tokens = tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=max_length,
        )
        tokens['labels'] = examples['labels']
        return tokens

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)

    keep_cols = {'input_ids', 'attention_mask', 'labels'}
    drop_train = [col for col in train_dataset.column_names if col not in keep_cols]
    drop_test = [col for col in test_dataset.column_names if col not in keep_cols]
    if drop_train:
        train_dataset = train_dataset.remove_columns(drop_train)
    if drop_test:
        test_dataset = test_dataset.remove_columns(drop_test)

    train_dataset.set_format(type='torch')
    test_dataset.set_format(type='torch')

    return train_dataset, test_dataset

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    labels = np.array(labels)
    logits = np.array(logits)

    if labels.ndim > 1 and labels.shape[-1] > 1:
        probs = 1 / (1 + np.exp(-logits))
        predictions = (probs >= 0.5).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='micro', zero_division=0
        )
        accuracy = (predictions == labels).mean()
    elif len(np.unique(labels)) > 2:
        predictions = np.argmax(logits, axis=-1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='macro', zero_division=0
        )
    else:
        predictions = np.argmax(logits, axis=-1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='binary', zero_division=0
        )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

def train_bert_model(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    model_name: str = DEFAULT_BERT_MODEL,
    output_dir: str = BERT_MODEL_DIR,
    config: TrainingConfig = None,
    multi_label: bool = False,
    label_names: Optional[List[str]] = None,
) -> Dict[str, Any]:
    if not TRANSFORMERS_AVAILABLE:
        raise ImportError("transformers and torch are required for BERT training")

    if config is None:
        config = TrainingConfig()

    print(f"\n{'='*60}")
    print(f"Fine-tuning: {model_name}")
    print(f"{'='*60}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print(f"Using device: {device_name} ({device})")

    print("Loading tokenizer and model...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

    num_labels = len(label_names) if multi_label else len(set(train_df['labels'].tolist()))
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )
    if multi_label:
        model.config.problem_type = "multi_label_classification"
    model.to(device)

    print("Preparing datasets...")
    train_dataset, test_dataset = create_datasets(
        train_df, test_df, tokenizer, config.max_length
    )

    training_args = TrainingArguments(
        output_dir=os.path.join(output_dir, 'checkpoints'),
        learning_rate=config.learning_rate,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        num_train_epochs=config.num_epochs,
        weight_decay=config.weight_decay,
        eval_strategy=config.eval_strategy,
        save_strategy=config.save_strategy,
        logging_strategy=config.eval_strategy,
        logging_steps=config.logging_steps,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        warmup_ratio=config.warmup_ratio,
        load_best_model_at_end=config.load_best_model_at_end,
        metric_for_best_model=config.metric_for_best_model,
        greater_is_better=config.greater_is_better,
        save_total_limit=2,
        report_to='none',
        fp16=config.fp16 and torch.cuda.is_available(),
        bf16=config.bf16 and torch.cuda.is_available(),
        dataloader_num_workers=config.dataloader_num_workers,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print("\nStarting training...")
    train_result = trainer.train()

    print("\nEvaluating...")
    eval_result = trainer.evaluate()

    print(f"\n--- Final Results ---")
    print(f"Accuracy: {eval_result['eval_accuracy']:.4f}")
    print(f"Precision: {eval_result['eval_precision']:.4f}")
    print(f"Recall: {eval_result['eval_recall']:.4f}")
    print(f"F1 Score: {eval_result['eval_f1']:.4f}")

    os.makedirs(output_dir, exist_ok=True)
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    print(f"\nModel saved to: {output_dir}")

    results = {
        'pipeline_name': output_dir.split('/')[-1] if '/' in output_dir else output_dir,
        'model_name': model_name,
        'accuracy': eval_result['eval_accuracy'],
        'precision': eval_result['eval_precision'],
        'recall': eval_result['eval_recall'],
        'f1': eval_result['eval_f1'],
        'train_samples': len(train_df),
        'test_samples': len(test_df),
        'epochs': config.num_epochs,
        'batch_size': config.batch_size,
        'learning_rate': config.learning_rate,
        'label_names': label_names,
    }

    results_path = os.path.join(output_dir, 'training_results.json')
    with open(results_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    return results

# --- Main Execution for BERT Models with Preprocessing Pipelines ---

print_header("Train BERT Models with Preprocessing Pipelines (6 Pipelines)")

# 1. Load the training and test data
if not os.path.exists(TRAIN_CSV_PATH) or not os.path.exists(TEST_CSV_PATH):
    print("Error: Training or Test data not found. Please ensure Step 1 is completed.")
    sys.exit(1)

train_df_raw = pd.read_csv(TRAIN_CSV_PATH, encoding='utf-8-sig') # Specify encoding
test_df_raw = pd.read_csv(TEST_CSV_PATH, encoding='utf-8-sig')   # Specify encoding

X_train_raw = train_df_raw['text'].astype(str).tolist()
y_train = train_df_raw['label'].astype(int).values

X_test_raw = test_df_raw['text'].astype(str).tolist()
y_test = test_df_raw['label'].astype(int).values

# 2. Initialize an empty list to store results
bert_results_all = []

# If a previous run was interrupted, ensure `bert_results_all` is populated with existing results
# This is a robust way to continue if an interruption happened partway through the loop
for pipeline_name_prev, config_prev in PREPROCESSING_PIPELINES.items():
    current_bert_model_dir_prev = os.path.join(MODELS_DIR, f"bert_model_{pipeline_name_prev}")
    results_path_prev = os.path.join(current_bert_model_dir_prev, 'training_results.json')
    if os.path.exists(results_path_prev):
        with open(results_path_prev, 'r', encoding='utf-8') as f:
            bert_results_all.append(json.load(f))

# 3. Iterate through each pipeline
for pipeline_name, config in PREPROCESSING_PIPELINES.items():
    current_bert_model_dir = os.path.join(MODELS_DIR, f"bert_model_{pipeline_name}")
    results_path_current = os.path.join(current_bert_model_dir, 'training_results.json')

    if os.path.exists(results_path_current):
        print(f"\nSkipping {pipeline_name}: results already exist.")
        continue # Skip if results for this pipeline already exist

    print(f"\n{'#'*70}")
    print(f"Training BERT with {pipeline_name}: {config}")
    print(f"{'#'*70}")

    # 4. Preprocess X_train_raw and X_test_raw
    X_train_processed = [preprocess_text_to_string(text, config) for text in X_train_raw]
    X_test_processed = [preprocess_text_to_string(text, config) for text in X_test_raw]

    # 5. Create new pandas DataFrames
    train_df_processed = pd.DataFrame({'text': X_train_processed, 'labels': y_train})
    test_df_processed = pd.DataFrame({'text': X_test_processed, 'labels': y_test})

    # 6. Define a unique output_dir for each BERT model
    os.makedirs(current_bert_model_dir, exist_ok=True)

    # 7. Call the train_bert_model function
    training_config_instance = TrainingConfig() # Use default or specific config for BERT
    label_names = sorted(np.unique(y_train).tolist()) # Get unique labels from y_train

    try:
        result = train_bert_model(
            train_df=train_df_processed,
            test_df=test_df_processed,
            model_name=DEFAULT_BERT_MODEL,
            output_dir=current_bert_model_dir,
            config=training_config_instance,
            label_names=label_names
        )
        # Add pipeline config string to result for summary table
        result['pipeline_config_str'] = str(config)
        # 8. Append the returned results
        bert_results_all.append(result)
    except Exception as e:
        print(f"Error training BERT for {pipeline_name}: {e}")
        bert_results_all.append({
            'pipeline_name': pipeline_name,
            'pipeline_config_str': str(config),
            'model_name': DEFAULT_BERT_MODEL,
            'accuracy': 0.0,
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0,
            'error': str(e)
        })

# 9. After the loop, convert bert_results_all into a pandas DataFrame and save it
bert_results_df = pd.DataFrame(bert_results_all)
summary_output_path = os.path.join(MODELS_DIR, 'bert_results_summary.csv')
bert_results_df.to_csv(summary_output_path, index=False, encoding='utf-8-sig')

print(f"\n{'='*70}")
print("BERT Training Summary Across All Pipelines")
print(f"{'='*70}")
print(bert_results_df.sort_values('f1', ascending=False).to_string())
print(f"Full summary saved to: {summary_output_path}")

: 